# Step 1: 敏感层识别（profiling + 逐层量化测 PPL）

**目标**：建立工业标准的「敏感层识别两步法」——(1) 用 forward hook 给每个 `Linear` 的输入激活做 per-channel 幅值统计，找离群点密集的层；(2) 逐层单独量化、其余层保持 FP16，测 PPL 跳升幅度，PPL 跳升者即**真正敏感**的层。这是 M3「layer fallback 调优」的第一步：你得先知道哪些层怕量化，才能决定回退哪些层。

**对应 OUTLINE 课时**：3.1 敏感层识别（~55 分钟）。


## 学完应能讲清（学完本节应能口头回答）

1. 敏感层识别的「两步法」分别是什么？为什么**两步都要做**（只做 profiling 够不够）？
2. profiling 测的是「激活幅值」，为什么激活幅值大就暗示这层对量化敏感？（离群点撑爆 scale）
3. 为什么「逐层单独量化测 PPL」要**保持其它层 FP16**？（隔离单层贡献，否则误差累积分不清谁的锅）
4. 为什么敏感度**与 scheme 相关**（FP8 的敏感层 ≠ INT4 的敏感层）？能不能拿一个 scheme 的结论去指导另一个？
5. PPL 作为敏感度判据比「激活幅值」好在哪？（PPL 是端到端任务指标，幅值只是代理信号）


In [ ]:
%%capture
import pathlib, os, re, math
import torch
import torch.nn as nn
import ipytest
ipytest.autoconfig()
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from transformers import Qwen2Config, Qwen2ForCausalLM, AutoTokenizer
from llmcompressor.modifiers.quantization import QuantizationModifier


In [ ]:
# Setup cell（双 env：每个子项目自带 pyproject.toml，但模块根是含 scripts/ + steps/ 的
# course/m3-tuning-eval/。子项目自己有 pyproject.toml 但无 scripts/，故别用 pyproject 判据）。
# 见 course/NOTEBOOK_CONVENTIONS.md 第 2 节 + 本模块设计 §4。
import pathlib

def _find_module_root(start):
    p = pathlib.Path(start).resolve()
    for cand in [p, *p.parents]:
        if (cand / "scripts").is_dir() and (cand / "steps").is_dir():
            return cand
    raise RuntimeError("找不到模块根（含 scripts/ + steps/ 的目录）")

MODULE_ROOT      = _find_module_root(pathlib.Path.cwd())
MODEL_DIR        = MODULE_ROOT / "models" / "Qwen2.5-7B-Instruct"
TINY_MODEL_DIR   = MODULE_ROOT / "models" / "Qwen2.5-0.5B-Instruct"
OUT_ROOT         = MODULE_ROOT / "out"; OUT_ROOT.mkdir(parents=True, exist_ok=True)
print("MODULE_ROOT =", MODULE_ROOT)
print("7B @", MODEL_DIR.exists(), "| 0.5B @", TINY_MODEL_DIR.exists(), "| out @", OUT_ROOT)


## 原理：什么是「敏感层」，为什么要找它

把整模型一刀切量化（M2 的做法）通常掉点。原因不是「所有层都掉一点」，而是**少数几层掉很多**——这几层叫**敏感层**。找到它们、单独回退到高精度，就能用很小的显存代价换回大部分精度。这是整个 M3 的核心工程哲学：**最少回退 → 最大精度**。

**两步法**（OUTLINE 3.1 工业标准）：

1. **激活异常 profiling（快、代理信号）**：给每个 `nn.Linear` 的输入挂 forward hook，按 channel 统计激活幅值（如 `max(|X|)` 或方差）。离群点密集的层 = 候选敏感层。
2. **逐层量化测 PPL（慢、最终判据）**：保持其它层全 FP16，**只把目标层量化**（如 INT8），测整模型 PPL。PPL 相对 FP16 基线跳升越多，这层越敏感。

**为什么两步都要做**（OUTLINE 3.1 易错点）：只看激活幅值会**误判**——有些层激活正常，但对量化误差的**累积**敏感（误差顺着 residual 流下去被放大）。profiling 给候选、PPL 给裁决。且敏感度**与 scheme 强相关**：FP8 的敏感层和 W4A16 的敏感层不一样，结论**不能跨 scheme 复用**（机制见下面「为什么敏感度与 scheme 强相关」）。

**为什么敏感度与 scheme 强相关**（把第 4 题的 why 讲透，不止复述结论）：不同 scheme 量化的是**不同对象**，所以「哪些层怕量化」由不同因素决定——

| scheme | 量化的对象 | 敏感度由什么决定 | 典型敏感层特征 |
|---|---|---|---|
**W4A16 / W8A16**（weight-only） | 只量化**权重** | 权重本身的**幅值分布**（离群权重通道） | 权重幅值离群严重的层 |
**W8A8 / FP8**（weight+activation） | 量化**权重和激活** | 权重幅值 **+ 激活离群** | 权重离群 **或** 激活离群（两者任一严重）的层 |

机制拆解：
- **W4A16 只碰权重**：激活全程 FP16，激活离群通道**完全不影响**敏感度。哪些层敏感，只看权重的 per-channel 幅值分布——某层权重幅值离群狠 → W4 把它量化就掉点 → 该层是 W4 敏感层。
- **W8A8 / FP8 还碰激活**：激活也要量化（见下面「scale-拉大因果链」），所以激活离群通道**直接决定**敏感度。某层权重幅值很正常、但激活里离群通道密集 → 它在 W4 下不敏感、在 W8A8 下却很敏感。

所以「W4 的敏感层集合」和「W8A8 的敏感层集合」是两套不同判断逻辑产出的结果：W4 只看权重、W8A8 权重+激活都要看。拿 W4 扫描出的敏感层去指导 W8A8 的 ignore 清单，会把「权重正常但激活离群」的层漏掉（或反之）。**结论：换一个 scheme，就得为它单独重跑一次敏感度扫描，不能用别的 scheme 的结论顶替。**

**为什么「激活幅值大」就暗示这层敏感**（scale-拉大因果链，把第 2 题的 why 画完整）：

激活幅值本身不直接伤模型，伤模型的是「**离群通道把整组/per-channel 的 scale 撑大**」这条因果链。以 per-channel 对称量化（INT8，`scale = max(|w|)/127`）为例，把同样的逻辑套到激活上：

```
某层有 128 个通道，其中 1 个通道激活幅值是 100，其余 127 个都在 1 附近
        │
        ▼  max(|激活|) = 100  →  scale = 100/127 ≈ 0.79
        │
        ▼  这 0.79 的 scale 管整组/per-channel 的所有通道
        │
        ▼  正常通道（幅值≈1）被这个被撑大的 scale 量化：
        │     量化步长 = 0.79，而正常通道值才 ~1 → 一个 step 就跨过去
        │     → 正常通道的有效量化粒度极粗（≈1 bit 的信息量）
        │     → 量化误差几乎全堆在这 127 个正常通道上
        ▼  结果：离群通道自己量化得挺准（它撑起了 scale），
              但其余 127 个正常通道被「连坐」，整层激活被严重扭曲 → 下游 PPL 跳
```

**关键认知**：离群通道不是「自己量化坏」，而是「**把 scale 拉大、连累正常通道的有效精度**」。所以 profiling 看 `max/mean`（离群比）比看裸 `max` 更准——离群比大 = 少数通道远超均值 = scale 被撑得越狠 = 正常通道被连坐越严重 = 越像敏感层。这也是为什么 s4 的 SmoothQuant 要先「平滑」离群点（把离群通道的幅值迁移一部分到权重上），本质就是**不让离群点撑爆 scale**。

> 注意：这条因果解释「幅值大→候选敏感」，但**不能保证**该层一定敏感（见上「为什么两步都要做」——可能幅值大但对累积不敏感）。profiling 给候选，PPL 给最终裁决。

### 端到端：敏感层识别在 layer fallback 全流程的哪一步

```
  全量化基线（M2，掉点）
      │
      ▼
  ┌─ s1 敏感层识别（本节）──── 产出：按敏感度排序的层清单
  │      │
  │      ▼
  ├─ s2/s3 经验法则 + ignore 语法 ── 把最敏感的层加进 ignore
  ├─ s4 mixed-precision ── 敏感层不只「全回退」，可配更高比特
  ├─ s5 Pareto ── 逐步加 ignore 找 PPL 恢复拐点
  └─ s7 评测 ── downstream 确认救回来了
```
本节产出**敏感度排序表**，是后面 s3/s4/s5 一切调优决策的输入。


## 亲手摸一摸：hook 拿到的激活统计长什么样

先建个 tiny Qwen2，跑几步随机 forward，看 hook 收集到的 per-channel 激活幅值——直观感受「有的 channel 幅值远大于其它」。


In [ ]:
## 摸一摸：给 tiny 模型的 q_proj 挂 hook，看输入激活的 per-channel 幅值
torch.manual_seed(0)
tiny = Qwen2ForCausalLM(Qwen2Config(
    num_hidden_layers=2, hidden_size=64, intermediate_size=128,
    num_attention_heads=4, num_key_value_heads=2, vocab_size=320, tie_word_embeddings=False))
tiny.eval()

saw = {}
def make_hook(name):
    def hook(_mod, inp, _out):
        x = inp[0].detach()                      # [batch, seq, in_features]
        per_channel = x.abs().flatten(0, 1).max(dim=0).values   # per-channel 最大幅值
        saw[name] = per_channel
    return hook

for n, m in tiny.named_modules():
    if isinstance(m, nn.Linear):
        m.register_forward_pre_hook(lambda _mod, inp, n=n: saw.update({n: inp[0].detach().abs().flatten(0,1).max(dim=0).values}))

with torch.no_grad():
    _ = tiny(input_ids=torch.randint(0, 320, (2, 16)))

name = 'model.layers.0.self_attn.q_proj'
v = saw[name]
print(f"{name}: in_features={v.numel()} 个 channel 的最大幅值")
print(f"  均值={v.mean():.3f}  最大={v.max():.3f}  最小={v.min():.3f}")
print(f"  最大/均值 = {v.max()/v.mean():.2f}  （越大越像有离群点 → 候选敏感层）")
print(f"  收集到幅值的 Linear 层数 = {len(saw)}")


## 本步填空

1. **`profile_activation_outliers(model, sample_inputs)`** —— 给模型所有 `Linear` 挂 hook，跑一遍 forward，返回每层 per-channel 最大幅值的「离群比」（max/mean）。**为什么这么设计（填前先想）**：单看 max 不够（大模型普遍幅值大），用 max/mean 归一化才能跨层比较「离群程度」。
2. **`per_layer_quant_ppl(model, tokenizer, target_layer, baseline_ppl)`**（判断型，提供 `compute_ppl` 工具）—— 保持其它层 FP16、只量化 `target_layer`，返回该层单独量化后的 PPL 相对基线的**上升比例**。**为什么这么设计**：要隔离单层贡献，必须只量化一层；PPL 是端到端判据，比幅值可靠。


In [ ]:
def profile_activation_outliers(model, sample_inputs):
    """给 model 所有 nn.Linear 挂 forward_pre_hook，跑一遍 sample_inputs forward，
    返回 {层名: 离群比} 字典。离群比 = max(|激活|).per-channel 的最大值 / 均值。

    为什么用 max/mean 而非裸 max：不同层幅值量级不同（attention vs mlp），
    归一化后才能跨层排序「谁离群更狠」。这一步是**快代理信号**——只前向、不量化。
    """
    # TODO: 1) 遍历 named_modules 找 nn.Linear；
    #       2) 给每层挂 forward_pre_hook，在 hook 里把 inp[0].abs().flatten(0,1).max(dim=0).values
    #          存进一个 dict（key=层名）；
    #       3) torch.no_grad() 下跑 model(**sample_inputs) 一次；
    #       4) 对每层算 离群比 = per_channel.max() / per_channel.mean()，返回 {层名: 比值}。
    #   提示：hook 的回调签名是 hook(module, inputs)->None，inputs 是 tuple，inputs[0] 是输入张量。
    raise NotImplementedError


In [ ]:
# 参考实现（reviewer 注入执行验证用；学员勿看，自己填）
def _profile_activation_outliers_ref(model, sample_inputs):
    stats = {}
    handles = []
    for name, mod in model.named_modules():
        if isinstance(mod, nn.Linear):
            def make_cb(nm):
                def cb(_mod, inp):
                    x = inp[0].detach().abs().flatten(0, 1)
                    stats[nm] = x.max(dim=0).values
                return cb
            handles.append(mod.register_forward_pre_hook(make_cb(name)))
    try:
        with torch.no_grad():
            model(**sample_inputs)
    finally:
        for h in handles:
            h.remove()
    return {n: float(v.max() / v.mean().clamp(min=1e-8)) for n, v in stats.items()}
profile_activation_outliers = _profile_activation_outliers_ref  # noqa


In [ ]:
# 工具：给定 logits 和 labels 算 cross-entropy PPL（tiny L2 用；7B L3 同样算）
def compute_ppl(logits, labels):
    """logits: [N, vocab], labels: [N]（已对齐：logits[i] 预测 labels[i]）。
    返回 PPL = exp(mean(cross_entropy))。"""
    loss = torch.nn.functional.cross_entropy(logits, labels)
    return float(torch.exp(loss).item())


def per_layer_quant_ppl(model, tokenizer, target_layer, baseline_ppl):
    """判断型：把 model 里**只有 target_layer 这一层**量化成 INT8（用 round-to-nearest
    朴素量化做隔离实验，不跑 GPTQ——隔离单层贡献、追求快），其余层保持 FP16，
    返回 PPL 相对 baseline_ppl 的上升比例（量化后PPL / baseline_ppl）。

    为什么这么设计（填前先想）：
    - 必须只量化一层：若同时量化多层，误差会沿 residual 累积，分不清是谁的锅。
    - 用朴素 RTN（权重除 scale、round 到 int8、再反量化回 float）即可，不跑 GPTQ——
      敏感度排序只需要相对比较，朴素量化的「相对敏感性」结论与 GPTQ 一致，但快几个量级。
    - input_ids 用与模型 vocab 一致的固定序列（如 torch.arange(16) % vocab_size），**不
      用 tokenizer 文本分词**——合成 tiny 模型（vocab=320）与真分词器词表不匹配，分词会
      得空序列、forward 报错（reshape 0 元素）。tokenizer 参数保留仅为兼容签名。
    """
    # TODO: 1) 拿到 model.get_submodule(target_layer)（一个 nn.Linear）；
    #       2) 备份原始 weight（tensor.clone()）；
    #       3) 对 weight 做 per-channel W8 对称量化再反量化回 float（模拟 INT8 量化误差）：
    #            scale = weight.abs().max(dim=1, keepdim=True).values / 127  （per-output-channel）
    #            q = torch.round(weight / scale).clamp(-127, 127)
    #            weight_q = (q * scale).to(weight.dtype)
    #          写回 layer.weight.data（注意 nn.Parameter.data）。
    #       4) 构造固定 input_ids（torch.arange(seq_len).unsqueeze(0) % vocab_size，seq_len=16，
    #          vocab_size 取自 model.config.vocab_size）——别用 tokenizer 文本分词（见上 why）；
    #          torch.no_grad() 下前向拿 logits，算 PPL（labels 用 input_ids 错位对齐）。
    #       5) finally 还原 layer.weight.data = 原始 weight（务必还原，否则污染模型）；
    #       6) 返回 ppl / baseline_ppl。
    #   提示：labels 用 input_ids 错位对齐（labels[1:] 对应 logits[:-1] 预测下一 token）。
    raise NotImplementedError

In [ ]:
# 参考实现（reviewer 注入执行验证用；学员勿看）
# tokenizer 无关：合成 tiny 模型（vocab=320）与真 0.5B 分词器词表不匹配——调
# tokenizer("...") 会分得空序列 → model(input_ids=[]) 报 RuntimeError（reshape 0 元素）。
# 故绕过分词、用与模型 vocab 一致的固定 input_ids 直接算 PPL（与 L2 的 _FakeTok 路径一致，最稳）。
def _per_layer_quant_ppl_ref(model, tokenizer, target_layer, baseline_ppl):
    layer = model.get_submodule(target_layer)
    orig = layer.weight.data.clone()
    vocab = getattr(getattr(model, "config", None), "vocab_size", None) or 320
    # 固定 input_ids（seq=16）：可复现、非空、id ∈ [0, vocab)。tokenizer 参数保留以兼容
    # 签名（L3 真模型可传真分词器，但这里仍用固定 ids，保证合成/tiny/真模型都稳）。
    input_ids = (torch.arange(16, dtype=torch.long).unsqueeze(0)) % vocab
    try:
        w = layer.weight.data
        scale = w.abs().max(dim=1, keepdim=True).values.clamp(min=1e-8) / 127.0
        q = torch.round(w / scale).clamp(-127, 127)
        layer.weight.data = (q * scale).to(w.dtype)
        with torch.no_grad():
            logits = model(input_ids=input_ids).logits[0]
        shift_logits = logits[:-1]
        shift_labels = input_ids[0, 1:]
        ppl = compute_ppl(shift_logits, shift_labels)
    finally:
        layer.weight.data = orig
    return ppl / baseline_ppl
per_layer_quant_ppl = _per_layer_quant_ppl_ref  # noqa

In [ ]:
def _tiny_with_inputs():
    torch.manual_seed(0)
    m = Qwen2ForCausalLM(Qwen2Config(
        num_hidden_layers=2, hidden_size=64, intermediate_size=128,
        num_attention_heads=4, num_key_value_heads=2, vocab_size=320, tie_word_embeddings=False))
    m.eval()
    inputs = {"input_ids": torch.randint(0, 320, (2, 16))}
    return m, inputs

def test_profile_activation_outliers_returns_all_linears():
    m, inputs = _tiny_with_inputs()
    out = profile_activation_outliers(m, inputs)
    linear_names = [n for n, _ in m.named_modules() if isinstance(_, nn.Linear)]
    assert set(out.keys()) == set(linear_names), "应覆盖每个 nn.Linear"
    for n, r in out.items():
        assert r >= 1.0, f"max/mean 应 >= 1（max>=mean），{n} 得 {r}"

def test_profile_activation_outliers_cleans_hooks():
    m, inputs = _tiny_with_inputs()
    before = sum(len(getattr(mod, "_forward_pre_hooks", {})) for _, mod in m.named_modules())
    profile_activation_outliers(m, inputs)
    after = sum(len(getattr(mod, "_forward_pre_hooks", {})) for _, mod in m.named_modules())
    assert after == before, "hook 跑完应 remove，否则泄露"

def test_per_layer_quant_ppl_is_ratio_and_restores_weight():
    m, _ = _tiny_with_inputs()
    target = "model.layers.0.mlp.down_proj"
    w_before = m.get_submodule(target).weight.data.clone()
    # tokenizer 不参与（参考实现用固定 input_ids），传 None 占位即可
    ratio = per_layer_quant_ppl(m, None, target, baseline_ppl=10.0)
    assert isinstance(ratio, float) and ratio > 0, "应返回正的上升比例"
    w_after = m.get_submodule(target).weight.data
    assert torch.allclose(w_before, w_after), "量化完应还原权重，不能污染模型"

# L1 必过守卫：ipytest.run 返回 pytest 退出码；非 0（有测试失败）→ 抛异常让 nbconvert 真挂。
# （用 ipytest.run() 而非 %%ipytest magic：magic 吞掉失败、exit_code 属性在本版不可靠，
#  见 NOTEBOOK_CONVENTIONS §验证机制。）
_ec = ipytest.run("-qq")
assert _ec == 0, f"L1 测试未全过（exit_code={_ec}），见上方 pytest 输出。"

## L2（tiny，CPU）：在 0.5B 规模跑通两步法

用真 `llmcompressor` 的 `QuantizationModifier`（但只 target 一层、`num_bits` 用 W8 模拟）跑通 profiling + 逐层排序。tiny 模型层少，肉眼能看到敏感度差异。


In [ ]:
## L2：tiny 模型逐层敏感度排序（profile 幅值 + 朴素逐层量化 PPL 代理）
torch.manual_seed(42)
tiny = Qwen2ForCausalLM(Qwen2Config(
    num_hidden_layers=4, hidden_size=64, intermediate_size=128,
    num_attention_heads=4, num_key_value_heads=2, vocab_size=320, tie_word_embeddings=False))
tiny.eval()

inputs = {"input_ids": torch.randint(0, 320, (2, 32))}
outlier_scores = profile_activation_outliers(tiny, inputs)

# 朴素量化基线 PPL（全 FP16）
with torch.no_grad():
    base_logits = tiny(**inputs).logits[0]
labels = inputs["input_ids"][0]
baseline_ppl = compute_ppl(base_logits[:-1], labels[1:])
print(f"FP16 基线 PPL = {baseline_ppl:.3f}")

# 逐层朴素量化（用参考实现的隔离逻辑：只量化一层测 PPL 比）
ppl_scores = {}
class _FakeTok:  # L2 不依赖真分词，用 input_ids 直接算 PPL
    def __call__(self, text, return_tensors=None):
        ids = inputs["input_ids"]
        return {"input_ids": ids}
for n in [k for k in outlier_scores if not k.startswith("lm_head")]:
    ppl_scores[n] = per_layer_quant_ppl(tiny, _FakeTok(), n, baseline_ppl)

ranked = sorted(ppl_scores.items(), key=lambda kv: -kv[1])
print("\n逐层敏感度（PPL 上升比，越大越敏感）前 5：")
for n, r in ranked[:5]:
    print(f"  {r:.3f}  {n}")
assert all(r >= 0.99 for _, r in ranked), "朴素量化至少不该让 PPL 下降"
print("\nL2 通过：两步法在 tiny 上跑通，产出敏感度排序表。")


## L3（H200，GPU 守卫）：真 Qwen2.5-7B SmoothQuant 敏感层扫描

在 7B 上跑 SmoothQuant（M2 的 recipe），先 profiling 找候选敏感层，再逐层量化测 PPL。这是 M3 后续 s3/s5 调优的真实输入。

> L3 双守卫：除 GPU 守卫外，reviewer 执行验证设 `SKIP_L3=1` 跳过真 7B（太慢）；真人/学员跑不设，L3 实证。


In [ ]:
import torch, os
def run_l3_sensitive_scan():
    from llmcompressor.modifiers.quantization import QuantizationModifier
    tok = AutoTokenizer.from_pretrained(MODEL_DIR)
    model = Qwen2ForCausalLM.from_pretrained(MODEL_DIR, torch_dtype=torch.float16, device_map="auto")
    model.eval()
    enc = tok("The future of AI depends on efficient inference.", return_tensors="pt").to(model.device)
    inputs = {"input_ids": enc["input_ids"]}
    scores = profile_activation_outliers(model, inputs)
    ranked = sorted(scores.items(), key=lambda kv: -kv[1])
    print("7B 敏感层 profiling top 8：")
    for n, r in ranked[:8]:
        print(f"  {r:.2f}  {n}")
    (OUT_ROOT / "s1_sensitive_rank.json").write_text(
        __import__("json").dumps(ranked[:16], indent=2))
    print("敏感层排序已存 out/s1_sensitive_rank.json（供 s3/s5 读）")

if torch.cuda.is_available() and not os.environ.get("SKIP_L3"):
    run_l3_sensitive_scan()
else:
    print("跳过 L3：无 GPU 或 SKIP_L3=1（CPU/CI 只验 L1+L2 代码逻辑）")


## 产物检查

L3 跑完会在 `out/` 产 `s1_sensitive_rank.json`（敏感层排序，供 s3 ignore / s5 Pareto 读）。下面读它、并画出 profiling 的敏感度分布。


In [ ]:
import json
rank_path = OUT_ROOT / "s1_sensitive_rank.json"
if rank_path.exists():
    rank = json.loads(rank_path.read_text())
    print(f"敏感层 top {min(8, len(rank))}：")
    for n, s in rank[:8]:
        print(f"  {s:.2f}  {n}")
    names = [r[0].split('.')[-1] + f"(L{r[0].split('.')[2]})" for r in rank[:10]]
    vals = [r[1] for r in rank[:10]]
    fig, ax = plt.subplots(figsize=(8,3)); ax.bar(range(len(vals)), vals); ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=45, ha='right')
    ax.set_ylabel("离群比 (max/mean)"); ax.set_title("7B 敏感层 profiling 分布"); fig.tight_layout(); plt.show()
else:
    print(f"{rank_path} 不存在（L3 未跑或被 SKIP_L3 跳过）。")
